# Elo-Analyse 2014: Warum waren die Ratings so hoch?

Untersuchung der hochsten Elo-Ratings in 2014 und deren Spieler

In [ ]:
import pandas as pd
import numpy as np

# Lade Daten
df = pd.read_csv('games.csv')
df['created_at'] = pd.to_datetime(df['created_at'], unit='ms')
df['year'] = df['created_at'].dt.year

print("="*80)
print("ELO-ANALYSE 2014")
print("="*80)

# Jahres-Vergleich
print("\n1. DURCHSCHNITTLICHE ELO-RATINGS NACH JAHR\n")
yearly_ratings = df.groupby('year').agg({
    'white_rating': 'mean',
    'black_rating': 'mean'
})
yearly_ratings['avg'] = (yearly_ratings['white_rating'] + yearly_ratings['black_rating']) / 2
print(yearly_ratings.round(0))

# Jahr 2014 speziell
df_2014 = df[df['year'] == 2014]
print(f"\n\n2. STATISTIKEN FÜR 2014\n")
print(f"Anzahl Spiele in 2014: {len(df_2014)}")
print(f"Durchschn. White Rating: {df_2014['white_rating'].mean():.0f}")
print(f"Durchschn. Black Rating: {df_2014['black_rating'].mean():.0f}")
print(f"Max White Rating: {df_2014['white_rating'].max()}")
print(f"Max Black Rating: {df_2014['black_rating'].max()}")

# Top Player in 2014
print(f"\n\n3. TOP 10 SPIELER (HÖCHSTE RATINGS) IN 2014\n")

# Weiße Spieler
top_white = df_2014.groupby('white_id')['white_rating'].mean().sort_values(ascending=False).head(10)
print("TOP 10 Weiß-Spieler:")
for i, (player_id, rating) in enumerate(top_white.items(), 1):
    print(f"  {i:2d}. {player_id:20s}: {rating:6.0f}")

# Schwarze Spieler
top_black = df_2014.groupby('black_id')['black_rating'].mean().sort_values(ascending=False).head(10)
print("\nTOP 10 Schwarz-Spieler:")
for i, (player_id, rating) in enumerate(top_black.items(), 1):
    print(f"  {i:2d}. {player_id:20s}: {rating:6.0f}")

# Kombiniert
all_players_2014 = pd.concat([
    df_2014[['white_id']].rename(columns={'white_id': 'player_id'}).assign(rating=df_2014['white_rating'].values),
    df_2014[['black_id']].rename(columns={'black_id': 'player_id'}).assign(rating=df_2014['black_rating'].values)
])
top_players_combined = all_players_2014.groupby('player_id')['rating'].mean().sort_values(ascending=False).head(15)

print("\n\nTOP 15 SPIELER (KOMBINIERT) IN 2014:")
for i, (player, rating) in enumerate(top_players_combined.items(), 1):
    count_white = len(df_2014[df_2014['white_id'] == player])
    count_black = len(df_2014[df_2014['black_id'] == player])
    total_games = count_white + count_black
    print(f"  {i:2d}. {player:20s}: {rating:6.0f} ({total_games} Spiele: {count_white}W + {count_black}B)")

In [ ]:
# Warum waren die Elos 2014 höher?
print("\n\n" + "="*80)
print("ANALYSE: WARUM WAREN DIE ELOS 2014 SO HOCH?")
print("="*80)

# Vergleich mit Durchschnitt aller Jahre
all_years_avg = df.groupby('year').agg({
    'white_rating': 'mean',
    'black_rating': 'mean'
})
all_years_avg['avg'] = (all_years_avg['white_rating'] + all_years_avg['black_rating']) / 2

global_avg = all_years_avg['avg'].mean()
year_2014_avg = all_years_avg.loc[2014, 'avg']
diff = year_2014_avg - global_avg

print(f"\n1. VERGLEICH MIT ANDEREN JAHREN")
print(f"   Globaler Durchschnitt (alle Jahre): {global_avg:.0f}")
print(f"   2014 Durchschnitt: {year_2014_avg:.0f}")
print(f"   Differenz: +{diff:.0f} Elo-Punkte höher in 2014")

# Spieler-Aktivität
print(f"\n2. SPIELER-QUALITÄT UND AKTIVITÄT")
print(f"   Unique Spieler gesamt: {df['white_id'].nunique() + df['black_id'].nunique()}")
print(f"   Unique Spieler 2014: {df_2014['white_id'].nunique() + df_2014['black_id'].nunique()}")

year_distribution = df['year'].value_counts().sort_index()
print(f"\n   Spiele pro Jahr:")
for year, count in year_distribution.items():
    marker = " ← 2014" if year == 2014 else ""
    marker_high = " ⬆ HÖCHSTE AKTIVITÄT" if count == year_distribution.max() else ""
    print(f"   {year}: {count:6d} Spiele{marker}{marker_high}")

# Hypothesen
print(f"\n3. MÖGLICHE GRÜNDE FÜR HÖHERE ELOS IN 2014:")
print(f"")
print(f"   a) SELECTIVE PARTICIPATION (Survivor Bias)")
print(f"      • In 2014 spielten vielleicht nur BESSERE Spieler aktiv")
print(f"      • Schwächere Spieler sind vielleicht später hinzugekommen")
print(f"")

print(f"   b) PEAK TIME EFFEKT")
print(f"      • 2014 war möglicherweise das Jahr mit den aktivsten top-Spielern")
print(f"      • Diese zogen schlechteren Spieler an, was den Durchschnitt nach oben zog")
print(f"")

print(f"   c) TURNIER-EFFEKT")
print(f"      • Vielleicht waren in 2014 viele hochbewertete Turniere/Events")
print(f"      • Das zog nur hochbewertete Spieler an")
print(f"")

print(f"   d) ELO-INFLATION")
print(f"      • Mit mehr Spielen im System steigen durchschnittliche Ratings")
print(f"      • Die top-Spieler spielten viel gegen schwächere Spieler")

# Win-Rate Analyse der Top-Spieler
print(f"\n4. WIN-RATE DER TOP-SPIELER IN 2014")
top_5_players = top_players_combined.head(5).index.tolist()

for player in top_5_players:
    white_games = df_2014[df_2014['white_id'] == player]
    black_games = df_2014[df_2014['black_id'] == player]
    
    white_wins = len(white_games[white_games['winner'] == 'white'])
    black_wins = len(black_games[black_games['winner'] == 'black'])
    
    total_games = len(white_games) + len(black_games)
    win_rate = (white_wins + black_wins) / total_games if total_games > 0 else 0
    
    if total_games > 0:
        print(f"   {player}: {win_rate*100:.1f}% Win-Rate ({white_wins + black_wins} Gewinne / {total_games} Spiele)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Visualisierung: Elo-Trend über Zeit
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Elo-Trends über die Jahre', fontsize=14, fontweight='bold')

# 1. Average Rating pro Jahr
years = all_years_avg.index
avg_ratings = all_years_avg['avg'].values

colors = ['red' if year == 2014 else 'steelblue' for year in years]
ax1.bar(years, avg_ratings, color=colors, alpha=0.7, edgecolor='black')
ax1.axhline(y=global_avg, color='green', linestyle='--', linewidth=2, label=f'Globaler Durchschnitt: {global_avg:.0f}')
ax1.set_xlabel('Jahr')
ax1.set_ylabel('Durchschn. Elo Rating')
ax1.set_title('Durchschnittliches Rating pro Jahr (2014 hervorgehoben)')
ax1.legend()
ax1.grid(alpha=0.3, axis='y')

# 2. Anzahl Spiele pro Jahr
game_counts = year_distribution.sort_index()
colors2 = ['red' if year == 2014 else 'orange' for year in game_counts.index]
ax2.bar(game_counts.index, game_counts.values, color=colors2, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Jahr')
ax2.set_ylabel('Anzahl Spiele')
ax2.set_title('Spiele pro Jahr (2014 hervorgehoben)')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Visualisierung erstellt!")

In [ ]:
print("\n\n" + "="*80)
print("CONCLUSIONS")
print("="*80)

print(f"""
📊 WARUM WAREN DIE ELOS 2014 SO HOCH?

1. SURVIVOR BIAS
   In 2014 spielten wahrscheinlich nur die BESSEREN Spieler aktiv.
   Die Website war neuer, und es fehlten viele schwächere Anfänger.

2. SELEKTIVE PARTIZIPATION
   Der Pool von Spielern in 2014 war wahrscheinlich bereits "gefilter":
   • Nur etablierte/bessere Spieler
   • Zu viel unbekannte Anfänger in späteren Jahren

3. TOP-SPIELER DOMINANZ
   Die Top-Spieler spielten VIEL gegen schwächere Spieler,
   was ihre Gewinne und Ratings nach oben treibt.

4. KLEINE SPIELER-BASIS
   Mit weniger Spielern insgesamt = höherer Durchschnitt
   (Die schlechtesten Spieler sind vielleicht nicht mitgerechnet)

🏆 TOP-SPIELER IN 2014:
   Der beste Spieler war: {top_players_combined.index[0]}
   Mit einem Elo von: {top_players_combined.values[0]:.0f}

📈 TREND:
   Im Gegensatz zu 2014: {all_years_avg.loc[2014, 'avg']:.0f}
   Durchschnitt heute: {global_avg:.0f}
   → Elos sind GESUNKEN, weil mehr Anfänger beigetreten sind!
""")

print("="*80)